# Train Ulang Detektor Plat — YOLO26n + Dataset Roboflow
Alur: **dataset Roboflow → fine-tune YOLO26n (640 px) → evaluasi test → export ONNX untuk Mini PC**

- Runtime: **GPU (L4/T4)** — *Runtime → Change runtime type*.
- API key Roboflow disimpan di **Colab Secrets** (ikon 🔑 di sidebar) dengan nama `ROBOFLOW_API_KEY` + aktifkan *Notebook access*.
- Semua hasil (bobot, grafik, ONNX) disimpan di Google Drive, jadi aman bila sesi Colab terputus.
- OCR tidak dilatih: Mini PC tetap memakai `plate_rec.onnx` + `plate_rec_dict.txt` yang lama.

## 0. Pengaturan

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ===== UBAH DI SINI BILA PERLU =====
RF_WORKSPACE = 'muhammads-workspace-a5wiz'
RF_PROJECT   = 'alpr-a7-lfttt'
RF_VERSION   = 2                 # naikkan bila membuat versi dataset baru di Roboflow
IMGSZ, EPOCHS, PATIENCE, BATCH = 640, 150, 30, 16
ROOT = '/content/drive/MyDrive/0 - KEPSTON/alpr_26n_rf'
# ===================================

import os
RUN_NAME = f'yolo26n_v{RF_VERSION}'
os.makedirs(ROOT, exist_ok=True)
!pip install -q -U "ultralytics>=8.4.0" roboflow onnx onnxslim onnxruntime

## 1. Unduh dataset dari Roboflow

In [ ]:
import yaml
from pathlib import Path
from roboflow import Roboflow

def roboflow_key():
    try:
        from google.colab import userdata
        return userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        from getpass import getpass
        return getpass('Roboflow API key: ').strip()

rf = Roboflow(api_key=roboflow_key())
ds = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download(
    'yolo26', location=f'/content/ds_v{RF_VERSION}', overwrite=False)
DS = Path(ds.location)

# Rapikan data.yaml: path absolut, test dipakai bila ada (kalau tidak, pakai valid)
d = yaml.safe_load(open(DS / 'data.yaml'))
TEST_SPLIT = 'test' if (DS / 'test/images').exists() else 'valid'
d.update(path=str(DS), train='train/images', val='valid/images', test=f'{TEST_SPLIT}/images')
yaml.safe_dump(d, open(DS / 'data.yaml', 'w'))
DATA = str(DS / 'data.yaml')

print('Kelas:', d['names'])
for s in ['train', 'valid', 'test']:
    n = len(list((DS / s / 'images').glob('*'))) if (DS / s / 'images').exists() else 0
    print(f'{s:6}: {n} gambar')

## 2. Fine-tune YOLO26n
Mulai dari `yolo26n.pt` (pretrained COCO). Berhenti otomatis bila `PATIENCE` epoch tidak membaik.
Bila sesi terputus, **jalankan sel ini lagi**: training dilanjutkan dari `last.pt` di Drive.

In [ ]:
import shutil
from ultralytics import YOLO

RUN  = Path(ROOT) / 'runs' / RUN_NAME
BEST = Path(ROOT) / f'{RUN_NAME}_best.pt'
last = RUN / 'weights' / 'last.pt'

if BEST.exists():
    print('Model sudah ada, training dilewati:', BEST)
elif last.exists():
    YOLO(str(last)).train(resume=True)
else:
    YOLO('yolo26n.pt').train(data=DATA, imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
                             seed=42, project=str(RUN.parent), name=RUN.name, exist_ok=True)
if not BEST.exists():
    shutil.copy(RUN / 'weights' / 'best.pt', BEST)

import matplotlib.pyplot as plt
plt.figure(figsize=(16, 8)); plt.imshow(plt.imread(RUN / 'results.png')); plt.axis('off'); plt.show()

## 3. Evaluasi pada data test
Metrik standar (mAP, precision, recall) + **confusion matrix** dihitung sendiri pada conf 0,25 dan 0,5 (IoU ≥ 0,5),
karena `val()` bawaan memakai conf 0,001 sehingga FP terlihat sangat banyak.

In [ ]:
model = YOLO(str(BEST))
m = model.val(data=DATA, split='test', imgsz=IMGSZ, verbose=False, plots=False)
P, R = m.box.mp, m.box.mr
print(f'Test | mAP50 = {m.box.map50:.3f} | mAP50-95 = {m.box.map:.3f} | '
      f'precision = {P:.3f} | recall = {R:.3f} | F1 = {2*P*R/(P+R+1e-9):.3f}')

In [ ]:
import numpy as np, cv2

def iou(a, b):
    x1, y1 = np.maximum(a[:, None, :2], b[None, :, :2]).transpose(2, 0, 1)
    x2, y2 = np.minimum(a[:, None, 2:], b[None, :, 2:]).transpose(2, 0, 1)
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)

def load_gt(img_path, w, h):
    lbl = img_path.parent.parent / 'labels' / f'{img_path.stem}.txt'
    rows = [list(map(float, l.split()[1:5])) for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
    return np.array([[(cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h] for cx, cy, bw, bh in rows]).reshape(-1, 4)

test_imgs = sorted((DS / TEST_SPLIT / 'images').glob('*'))
preds = {}
for p in test_imgs:
    r = model.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    preds[p] = (r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.orig_shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, conf in zip(axes, [0.25, 0.5]):
    TP = FP = FN = 0
    for p, (box, sc, (h, w)) in preds.items():
        pb, gt = box[sc >= conf], load_gt(p, w, h)
        matched = set()
        if len(pb) and len(gt):
            M_ = iou(pb, gt)
            for i in np.argsort(-sc[sc >= conf]):
                j = int(M_[i].argmax())
                if M_[i, j] >= 0.5 and j not in matched:
                    matched.add(j); TP += 1
                else:
                    FP += 1
        else:
            FP += len(pb)
        FN += len(gt) - len(matched)
    prec, rec = TP / (TP + FP + 1e-9), TP / (TP + FN + 1e-9)
    print(f'conf {conf}: TP={TP} FP={FP} FN={FN} | precision={prec:.3f} recall={rec:.3f} F1={2*prec*rec/(prec+rec+1e-9):.3f}')
    cm = np.array([[TP, FN], [FP, 0]])
    ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, '-' if (i, j) == (1, 1) else v, ha='center', va='center', fontsize=16,
                color='white' if v > cm.max() / 2 else 'black')
    ax.set_xticks([0, 1], ['plate', 'background']); ax.set_yticks([0, 1], ['plate', 'background'])
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Sebenarnya'); ax.set_title(f'Confusion matrix (conf {conf})')
plt.tight_layout(); plt.savefig(f'{ROOT}/{RUN_NAME}_confusion.png', dpi=150); plt.show()

### Contoh prediksi pada data test

In [ ]:
import random
random.seed(0)
sample = random.sample(test_imgs, min(8, len(test_imgs)))
plt.figure(figsize=(20, 9))
for k, p in enumerate(sample):
    plt.subplot(2, 4, k + 1)
    plt.imshow(model.predict(str(p), imgsz=IMGSZ, conf=0.5, verbose=False)[0].plot()[:, :, ::-1])
    plt.axis('off'); plt.title(p.name[:30], fontsize=8)
plt.tight_layout(); plt.show()

## 4. Export ONNX untuk Mini PC
Menghasilkan `plate_det.onnx` (input 640, keluaran end-to-end `[1, 300, 6]`) di folder Drive `minipc_<run>/`.
Di Mini PC: taruh bersama `plate_rec.onnx` + `plate_rec_dict.txt` lama, lalu arahkan `ALPR_MODEL_DIR` ke folder itu.

In [ ]:
import onnxruntime as ort
EXPORT = Path(ROOT) / f'minipc_{RUN_NAME}'
EXPORT.mkdir(exist_ok=True)

f = model.export(format='onnx', imgsz=IMGSZ, opset=17, simplify=True, dynamic=False)
shutil.copy(f, EXPORT / 'plate_det.onnx')

sess = ort.InferenceSession(str(EXPORT / 'plate_det.onnx'), providers=['CPUExecutionProvider'])
print('Input :', sess.get_inputs()[0].shape)
print('Output:', sess.get_outputs()[0].shape, '(harus [1, 300, 6])')

# Cek hasil ONNX sama dengan model .pt
onnx_model = YOLO(str(EXPORT / 'plate_det.onnx'), task='detect')
same = sum(len(model.predict(str(p), imgsz=IMGSZ, conf=0.5, verbose=False)[0].boxes) ==
           len(onnx_model.predict(str(p), imgsz=IMGSZ, conf=0.5, verbose=False)[0].boxes) for p in test_imgs[:50])
print(f'Jumlah deteksi ONNX = .pt pada {same}/{min(50, len(test_imgs))} gambar test')
print('Tersimpan:', EXPORT / 'plate_det.onnx', f'({(EXPORT / "plate_det.onnx").stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Unduh langsung (hanya jalan di Colab browser; di VS Code ambil saja dari Google Drive)
from google.colab import files
files.download(str(EXPORT / 'plate_det.onnx'))